# PMITC v1.2 benchmarks
Results of `benchmarks/pmitc_bench.py` (leave-one-case-out plus the time split, three label sets, seed 192). Headline tasks T1s, T2s, T2-open; trained on label_standard, label_strict evaluation-only, then a quick end-to-end re-run of T2 on the released files.

In [1]:
import json, sys, pandas as pd
sys.path.insert(0, '../benchmarks'); import pmitc_bench as B
res = json.load(open('../benchmarks/baselines.json'))
rows = []
for s in B.SETS:
    for t in ['T1s','T2s','T2open','T1','T2','T3','T3b']:
        for m, r in res['precise'][s][t]['summary'].items():
            rows.append(dict(label_set=s, task=t, model=m, loco_mean=r['loco_mean'], loco_sd=r['loco_sd'], n_folds=r['n_loco_folds'], time_split=r['time_split']))
pd.DataFrame(rows).pivot_table(index=['task','model'], columns='label_set', values='loco_mean')

label_set                       label_broad  label_standard  label_strict
task   model                                                             
T1     account_age_heuristic         0.0507          0.0507        0.0054
       gate1_rule                    0.2328          0.2328        0.3616
       logistic_regression           0.3772          0.3627        0.2137
       prevalence                    0.0036          0.0036        0.0007
T1s    account_age_heuristic         0.4254          0.4254        0.5454
       gate1_rule                    0.4231          0.4231        0.5184
       gradient_boosting_tuned       0.4880          0.4761        0.5511
       logistic_regression           0.5150          0.5136        0.4565
       prevalence                    0.1340          0.1340        0.1614
T2     account_age_heuristic         0.7996          0.7996        0.7772
       gate1_rule                    0.7314          0.7314        0.7601
       logistic_regression           0.7931          0.7614        0.6768
       prevalence                    0.5000          0.5000        0.5000
T2open account_age_heuristic         0.7951          0.7951        0.7715
       gate1_rule                    0.7241          0.7241        0.7537
       gradient_boosting_tuned       0.8470          0.8469        0.8184
       logistic_regression           0.7676          0.7331        0.6289
       prevalence                    0.5000          0.5000        0.5000
T2s    account_age_heuristic         0.5676          0.5676        0.4955
       gate1_rule                    0.7555          0.7555        0.8404
       gradient_boosting_tuned       0.7630          0.7146        0.6844
       logistic_regression           0.7534          0.7029        0.4897
       prevalence                    0.5000          0.5000        0.5000
T3     gate1_rule                    0.7538          0.7538        0.7633
       gradient_boosting_tuned       0.8608          0.8722        0.9673
       logistic_regression           0.8826          0.8834        0.9219
       prevalence                    0.5000          0.5000        0.5000
T3b    gate1_rule                    0.5295          0.5295        0.4557
       gradient_boosting_tuned       0.8647          0.9581        0.9667
       logistic_regression           0.8814          0.8727        0.9252
       prevalence                    0.5000          0.5000        0.5000

In [2]:
pd.DataFrame(rows)[lambda d: d.task.isin(['T1s','T2s','T2open'])]

,label_set,task,model,loco_mean,loco_sd,n_folds,time_split
0,label_strict,T1s,account_age_heuristic,0.5454,0.3746,4,0.2830
1,label_strict,T1s,gate1_rule,0.5184,0.3722,4,0.2770
2,label_strict,T1s,gradient_boosting_tuned,0.5511,0.3805,4,0.3157
3,label_strict,T1s,logistic_regression,0.4565,0.2668,4,0.3285
4,label_strict,T1s,prevalence,0.1614,0.2463,4,0.1985
5,label_strict,T2s,account_age_heuristic,0.4955,0.1163,4,0.5817
6,label_strict,T2s,gate1_rule,0.8404,0.1487,4,0.8034
7,label_strict,T2s,gradient_boosting_tuned,0.6844,0.1361,4,0.6823
8,label_strict,T2s,logistic_regression,0.4897,0.1799,4,0.7966
9,label_strict,T2s,prevalence,0.5000,0.0000,4,0.5000


In [3]:
# known shortcuts and leakage lists
res['known_shortcuts']

{'T2.share_on_this_ladder_market': 'controls are sampled one market at a time, so the value is always 1 for a control (excluded)',
 'T2.hours_first_buy_to_tnews': 'controls are matched on first-buy time within 24-72 h, so this cannot separate classes much (kept; check its univariate AUC)',
 'T3b.*': 'placebo markets are crypto-price and weather markets: volume shape, price path and buyer mix differ by market type, not by informed trading',
 'T1.log_buy_usdc_pre / log_market_usdc_pre / log_n_buy_fills_pre': 'selection by the sources, not construction: sources flag large bets, so pre-cut size alone ranks positives highly (univariate AUC 0.93-0.98). A detector that learns size learns what the press notices.',
 'T2.prior_net_long_win_rate / n_prior_markets': 'partly construction: control pools exclude Orca- and Inca-flagged wallets (A6.10), which removes many strong past traders from the controls; and partly selection (sources flag fresh wallets). Univariate AUC 0.82-0.88.',
 'T3.log_usdc_

In [4]:
# end-to-end re-run on the released files: T2s (age-matched controls), evaluated on label_strict, trained on label_standard; no gradient boosting
splits = pd.read_csv('../benchmarks/splits.csv')[['case_id','split']]
r = B.t2('../data', splits, 'precise', 'label_strict', tune=False, quick=True, design='age_matched')
r['summary']

{'account_age_heuristic': {'loco_mean': 0.4955,
  'loco_sd': 0.1163,
  'loco_min': 0.398,
  'loco_max': 0.6838,
  'n_loco_folds': 4,
  'time_split': 0.5817},
 'gate1_rule': {'loco_mean': 0.8404,
  'loco_sd': 0.1487,
  'loco_min': 0.6398,
  'loco_max': 1.0,
  'n_loco_folds': 4,
  'time_split': 0.8034},
 'logistic_regression': {'loco_mean': 0.4897,
  'loco_sd': 0.1799,
  'loco_min': 0.2,
  'loco_max': 0.6774,
  'n_loco_folds': 4,
  'time_split': 0.7966},
 'prevalence': {'loco_mean': 0.5,
  'loco_sd': 0.0,
  'loco_min': 0.5,
  'loco_max': 0.5,
  'n_loco_folds': 4,
  'time_split': 0.5}}